# Fase inicial: Extracción de métricas de fichero de audio.

## Características de esta versión

Este fichero tiene el propósito de extraer métricas de los ficheros de audio, para su posterior uso en entrenamiento de una red neuronal. Una de las columnas del dataset generado será si es real (bonafide) o fake (spoon)

En esta versión creamos un dataset con una linea por audio, obteniendo las medias de cada métrica en cada ventana temporal de 20 - 40 ms (librosa). Es decir. Obtenemos las métricas por cada ventana temporal, y luego hacemos un promedio. Así generamos un csv con una fila por audio, y cada una de las métricas como columnas.

In [15]:
import pandas as pd

# Cargar las etiquetas del dataset ASVspoof
ruta_protocolo = '../data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
columnas = ['speaker_id', 'file_name', 'system_id', 'attack_id', 'label']
df_etiquetas = pd.read_csv(ruta_protocolo, sep=' ', names=columnas)


# Filtrar solo los nombres de archivo y si son reales o falsos
df_etiquetas = df_etiquetas[['file_name', 'label']]

print(df_etiquetas.head())

      file_name     label
0  LA_T_1138215  bonafide
1  LA_T_1271820  bonafide
2  LA_T_1272637  bonafide
3  LA_T_1276960  bonafide
4  LA_T_1341447  bonafide


## Prueba 1

Esta es basada en el artículo de Mael Fabien - https://maelfabien.github.io/machinelearning/Speech9/#6-mel-frequency-cepstral-coefficients-mfcc

El artículo menciona 14 puntos, pero dejé fuera algunas cosas por eficiencia computacional y relevancia:

* Tempogram (Punto 9) y Polyfeatures (Punto 8): Son características orientadas casi exclusivamente al Análisis de Información Musical (MIR), para detectar acordes o ritmos musicales. En detección de voz falsa aportan mucho "ruido" computacional y poco valor.

* Frecuencia Fundamental o Pitch (Punto 11): Es valiosísima para detectar emociones o distinguir género, pero extraerla con precisión en Python requiere algoritmos iterativos (como YIN o pYIN) que son computacionalmente costosos. Ralentizaría drásticamente tu procesamiento de miles de audios en esta primera prueba.

* Jitter (Punto 12): Requiere tener primero una extracción perfecta de la Frecuencia Fundamental, por lo que entra en la misma categoría de coste computacional.

In [16]:
import librosa
import numpy as np
import pandas as pd
import os

def extract_features(file_path):
    # Intentar cargar el archivo de audio con manejo de errores
    try:
        # sr=None preserva la tasa de muestreo original del archivo
        y, sr = librosa.load(file_path, sr=None)
    except Exception as e:
        print(f"Error al procesar {file_path}: {e}")
        return None

    features = {}
    
    # 1. Estadisticas basicas de la forma de onda
    features['signal_mean'] = np.mean(y)
    features['signal_std'] = np.std(y)
    
    # 2. Energia (Root Mean Square Energy)
    rmse = librosa.feature.rms(y=y)
    features['rmse_mean'] = np.mean(rmse)
    
    # 3. Zero-Crossing Rate (ZCR)
    zcr = librosa.feature.zero_crossing_rate(y)
    features['zcr_mean'] = np.mean(zcr)
    
    # 4. Tempo (BPM)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    # Dependiendo de la version de librosa, tempo puede ser un float o un array
    features['tempo_bpm'] = tempo[0] if isinstance(tempo, np.ndarray) else tempo
    
    # 5. MFCCs (13 coeficientes es el estandar mas comun para voz)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    # Colapsamos la matriz temporal calculando la media y desviacion de cada coeficiente
    for i in range(1, 14):
        features[f'mfcc_{i}_mean'] = np.mean(mfccs[i-1])
        features[f'mfcc_{i}_std'] = np.std(mfccs[i-1])
        
    # 6. Caracteristicas Espectrales
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    features['spectral_centroid_mean'] = np.mean(spectral_centroid)
    
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features['spectral_bandwidth_mean'] = np.mean(spectral_bandwidth)
    
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features['spectral_rolloff_mean'] = np.mean(spectral_rolloff)

    return features

def process_batch(file_list):
    all_features = []
    
    for file in file_list:
        # Verificacion de seguridad antes de procesar
        if not os.path.exists(file):
            print(f"Ruta no encontrada (saltando): {file}")
            continue
            
        print(f"Extrayendo caracteristicas de: {file}")
        file_features = extract_features(file)
        
        if file_features is not None:
            # Guardamos el nombre del archivo como identificador
            file_features['file_name'] = os.path.basename(file)
            all_features.append(file_features)
            
    # Si ningun archivo se pudo procesar, devolvemos un DataFrame vacio de forma segura
    if not all_features:
        print("Advertencia: No se extrajeron caracteristicas de ningun archivo.")
        return pd.DataFrame()
        
    # Convertir a un DataFrame de Pandas
    df = pd.DataFrame(all_features)
    
    # Reordenar columnas para que el nombre del archivo sea la primera
    cols = ['file_name'] + [c for c in df.columns if c != 'file_name']
    df = df[cols]
    
    return df

# --- Bloque de Ejecucion ---
if __name__ == "__main__":
    archivos_de_prueba = [
        '../data/LA/ASVspoof2019_LA_train/flac/LA_T_1000137.flac', 
        '../data/LA/ASVspoof2019_LA_train/flac/LA_T_1000406.flac'
    ] 
    
    print("Iniciando procesamiento...")
    df_resultados = process_batch(archivos_de_prueba)
    
    if not df_resultados.empty:
        print("\nExtraccion completada con exito. Muestra de los datos:")
        print(df_resultados.head())
    else:
        print("\nError: El DataFrame esta vacio. Revisa las rutas de tus archivos de audio.")
        # Util para debuggear: imprimir el directorio actual
        print(f"Directorio de ejecucion actual: {os.getcwd()}")
        
        # Opcional: Guardar los resultados para no tener que recalcularlos
        df_resultados.to_csv('caracteristicas_audios.csv', index=False)

Iniciando procesamiento...
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1000137.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1000406.flac

Extraccion completada con exito. Muestra de los datos:
           file_name   signal_mean  signal_std  rmse_mean  zcr_mean  \
0  LA_T_1000137.flac -3.661560e-07    0.110664   0.090804  0.109839   
1  LA_T_1000406.flac -1.152907e-06    0.121382   0.070201  0.093658   

    tempo_bpm  mfcc_1_mean  mfcc_1_std  mfcc_2_mean  mfcc_2_std  ...  \
0  170.454545  -247.769653  173.474854    77.113968   53.324604  ...   
1  133.928571  -329.794922  140.092133    72.277245   44.523670  ...   

   mfcc_10_std  mfcc_11_mean  mfcc_11_std  mfcc_12_mean  mfcc_12_std  \
0    18.878401     -8.750018    10.329152    -11.132873     8.926097   
1    12.055736    -16.890476    13.625458     -8.430140     7.405041   

   mfcc_13_mean  mfcc_13_std  spectral_centroid_mean  spectral_bandwidth_mean  \
0    -14.56416

In [17]:
import pandas as pd
import os

# 1. Definir las rutas relativas (basado en la estructura de tu proyecto)
ruta_protocolo = '../data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ruta_audios = '../data/LA/ASVspoof2019_LA_train/flac/'

# 2. Cargar el protocolo y limpiar los datos
print("Leyendo el archivo de protocolo...")
nombres_cols = ['speaker_id', 'file_name', 'system_id', 'attack_id', 'label']
df_protocolo = pd.read_csv(ruta_protocolo, sep=' ', names=nombres_cols)

# Quedarnos solo con lo util: nombre del archivo y si es real o fake
df_etiquetas = df_protocolo[['file_name', 'label']].copy()

# Anadir la extension .flac para que coincida con los archivos reales
df_etiquetas['file_name_ext'] = df_etiquetas['file_name'] + '.flac'

# Construir la ruta absoluta/relativa completa hacia cada audio
df_etiquetas['ruta_completa'] = df_etiquetas['file_name_ext'].apply(
    lambda x: os.path.join(ruta_audios, x)
)

# 3. Preparar la lista de archivos a procesar
archivos_a_procesar = df_etiquetas['ruta_completa'].tolist()

# CONTROL DE TIEMPO: Cambia este valor a None cuando quieras procesar los 25.000 audios
LIMITAR_A = 50 
if LIMITAR_A is not None:
    archivos_a_procesar = archivos_a_procesar[:LIMITAR_A]
    print(f"\nMODO PRUEBA: Solo se procesaran los primeros {LIMITAR_A} audios.")
else:
    print(f"\nMODO MASIVO: Se procesaran {len(archivos_a_procesar)} audios. Ve a por un cafe.")

# 4. Llamar a la funcion de la celda anterior
# (Asegurate de haber ejecutado la celda donde definiste process_batch)
df_features = process_batch(archivos_a_procesar)

if not df_features.empty:
    # 5. Unir (Merge) las caracteristicas extraidas con sus etiquetas reales/fakes
    print("\nUniendo caracteristicas con las etiquetas del protocolo...")
    
    # Hacemos un JOIN usando el nombre del archivo con extension
    df_final = pd.merge(
        df_features, 
        df_etiquetas[['file_name_ext', 'label']],
        left_on='file_name', 
        right_on='file_name_ext', 
        how='inner'
    )
    
    # Limpiar columnas redundantes despues del merge
    df_final = df_final.drop(columns=['file_name_ext'])
    
    # 6. Guardar el dataset final listo para entrenar modelos
    nombre_salida = 'dataset_caracteristicas_train.csv'
    df_final.to_csv(nombre_salida, index=False)
    
    print(f"\nExito. Dataset guardado como '{nombre_salida}'")
    print(f"Dimensiones de tu nuevo dataset: {df_final.shape}")
    print("\nDistribucion de clases procesadas:")
    print(df_final['label'].value_counts())
else:
    print("\nFallo en la extraccion. No se generara el archivo CSV.")

Leyendo el archivo de protocolo...

MODO PRUEBA: Solo se procesaran los primeros 50 audios.
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1138215.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1271820.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1272637.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1276960.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1341447.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1363611.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1596451.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1608170.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1684951.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac/LA_T_1699801.flac
Extrayendo caracteristicas de: ../data